In [ ]:
import control as ct
import numpy as np
import matplotlib.pyplot as plt

## Define a function to generate a PID controller TF

In [ ]:
def Pid(Kp=None, Ki=None, Kd=None, freq_lpf=None, units=None):
    """Create a PID controller continuous-time transfer function.

    Args:
        Kp: Proportional gain.
        Ki: Integral gain.
        Kd: Derivative gain.
        freq_lpf_Hz: Cutoff frequency of the low-pass filter on the derivative term (in Hz).

    Returns:
        A control.TransferFunction representing the PID controller.
    """
    # proportional term
    if Kp is not None:
        C_p = ct.zpk([], [], Kp)
    else:
        C_p = ct.zpk([], [], 0)

    # integral term
    if Ki is not None:
        C_i = ct.zpk([], 0, Ki)
    else:
        C_i = ct.zpk([], [], 0)

    # derivative term
    if Kd is not None:
        if freq_lpf is not None:  # check for low-pass filter frequency and units
            if units is not None:
                match units:
                    case "Hz":
                        freq_lpf_rad_per_s = freq_lpf
                    case "rad/s":
                        freq_lpf_rad_per_s = 2 * np.pi * freq_lpf
                    case _:
                        raise ValueError("'units' must either be 'Hz' or 'rad/s'")

                C_d = ct.zpk([], [-freq_lpf_rad_per_s], Kd * freq_lpf_rad_per_s)
            else:
                raise ValueError("'units' of frequency must be specified")
        else:
            raise ValueError("A value for Kd was specified, but 'freq_lpf' was not")
    else:
        C_d = ct.zpk([], [], 0)

    # connect transfer functions in parallel
    C_pid = C_p + C_i + C_d
    return C_pid

## Function to generate actuator TF

In [ ]:
def FirstOrderDelay(time_constant, gain=1.0):
    """Create a first-order delay continuous-time transfer function.

    Args:
        time_constant: Time constant of the first-order delay (seconds)
        gain (optional): Gain of the first-order delay transfer function.

    Returns:
        A control.TransferFunction representing the first-order delay.
    """
    return ct.TransferFunction([gain], [time_constant, 1])

In [ ]:
# test delay
time_constant = 0.5 # 0.4
A = FirstOrderDelay(time_constant)
ct.ss(A)

t, y = ct.forced_response(A, T=np.linspace(0, 4, 100), U=np.ones(100))

plt.plot(t, y)
plt.grid(True)

In [ ]:
C_pid = Pid(Kp=1, Ki=0.1, Kd=2, freq_lpf=10, units="Hz")
print(C_pid)

# C_p = Pid(Kp=2)
# print(f"C_p: {C_p}")

# C_i = Pid(Ki=2)
# print(f"C_i: {C_i}")

# C_d = Pid(Kd=2, freq_lpf=10, units="Hz")
# print(f"C_d: {C_d}")

# print(f"C_pi: {C_p + C_i}")

In [ ]:
# convert to continuous-time state space object
C_ct = ct.tf2ss(C_pid)
print(C_ct)

In [ ]:
# convert to descrete-time
C_dt = ct.c2d(C_ct, 1 / 100)
print(C_dt)
# notice how C and D stay the same

In [ ]:
t, y = ct.forced_response(C_pid, T=np.linspace(0, 2, 100), U=np.ones(100))
plt.plot(t, y)
plt.grid(True)

In [ ]:
# test the controller
setpoint = 0
x = 1
error = setpoint - x
ctrl_state_0 = np.zeros(len(C_dt.A))  # initialize controller state
u = np.array([error])

N = 200
ctrl_state_k = ctrl_state_0
print(f"control state:\t{ctrl_state_0}")
print(f"output:\t\t{0}")
print(f"input (error):\t{u}\n")
for i in range(N):
    # next controller state (x_kp1 = Ax + Bu)
    ctrl_state_kp1 = C_dt.A @ ctrl_state_k + C_dt.B @ u

    # output of controller (y = Cx + Du)
    y = C_dt.C @ ctrl_state_k + C_dt.D @ u

    print(f"control state:\t{ctrl_state_kp1}")
    print(f"output:\t\t{y}")
    print(f"input (error):\t{u}\n")

    # update state
    ctrl_state_k = ctrl_state_kp1

In [ ]:
### SCRATCH CAPSTONE WORK BELOW

import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('outputs', exist_ok=True)

# Load existing CSV to preserve the exact thrust curve
csv_path = os.path.join('outputs', 'thrust_data_t2_7s.csv')
data = np.loadtxt(csv_path, delimiter=',', skiprows=1)

t = data[:,0]
cmd = data[:,1]
thrust = data[:,2]

# Plot using loaded data and smaller arrowheads
plt.figure(figsize=(9,4.5))
plt.plot(t, thrust, label='Measured thrust', color='#1f77b4')
plt.plot(t, cmd, '--', label='Commanded thrust', color='#ff7f0e')
plt.xlabel('Time (s)')
plt.ylabel('Thrust (lbf)')
plt.title('(FAKE DATA) Example rocket engine injector test (hotfire) w\ith throttling')
plt.grid(alpha=0.3)
plt.legend()
plt.xlim(0, 7)
plt.ylim(-5, 90)

# Smaller horizontal double-arrow measuring 1 second between 2.0s command change and 3.0s settling
x_start = 2.0
x_end = 3.0
y_pos = 62
plt.annotate('', xy=(x_start, y_pos), xytext=(x_end, y_pos),
             arrowprops=dict(arrowstyle='<->', lw=2.0, color='black', mutation_scale=18))
plt.text((x_start + x_end)/2, y_pos + 3.0, '1 s', ha='center', va='bottom',
         fontsize=12, fontweight='bold')

# mark the 2.0s command change for clarity
plt.axvline(2.0, color='gray', lw=0.9, linestyle=':')

out_png = os.path.join('outputs', 'thrust_hotfire_t2_7s_smallerheads.png')
plt.tight_layout()
plt.savefig(out_png, dpi=150)
plt.close()

print('Saved:', out_png)
print('CSV used:', csv_path)
